# Семинар 7 [17-25-2]. Бонусное задание: исследование на симуляции (+5 баллов)

**ФИО:** ...
**Группа:** ...
**Вариант (номер в списке группы):** ...

### Как устроено задание

Пять этапов, каждый — **1 балл**. Код почти готов: в местах `# TODO` нужно дописать одну-три строки. **Баллы ставятся за интерпретацию**: за прогноз, сделанный *до* запуска, за объяснение того, что получилось, и за вывод своими словами.

В каждом этапе есть три текстовые ячейки:

- **Прогноз (до запуска)** — заполняется *до* того, как вы запустите код этапа. Неверный прогноз не снижает балл, если потом вы объяснили, почему он не сбылся.
- **Что получилось** — числа и графики, которые вы увидели.
- **Интерпретация** — почему так получилось, со ссылкой на формулы и условия их применения.

**Защита:** преподаватель может попросить изменить один параметр и до запуска предсказать, как изменится график. Ответ «так сказала нейросеть» баллов не приносит, а умение объяснить свой график — приносит.

Как работать в Colab: загрузите этот файл через **Файл → Загрузить блокнот**. Он сохранится на вашем Google Диске в папке «Colab Notebooks», дальше открывайте его оттуда. Для сдачи скачайте заполненный ноутбук: **Файл → Скачать → Скачать .ipynb**.

In [ ]:
VARIANT = 0   # ← впишите номер своего варианта (1–30)

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

assert 1 <= VARIANT <= 30, "Впишите номер варианта от 1 до 30"
rng = np.random.default_rng(1000 + VARIANT)
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Параметры вашего варианта
P_GERM = round(0.62 + 0.01 * ((VARIANT * 11) % 30), 2)   # всхожесть семян
N_BAG = 20 + 5 * (VARIANT % 5)                           # семян в пакете (этап 1)
EPS = [0.005, 0.01, 0.02][VARIANT % 3]                   # допустимая ошибка Пуассона (этап 3)
print(f"Вариант {VARIANT}: всхожесть p = {P_GERM}, семян в пакете n = {N_BAG}, порог ошибки = {EPS}")

In [ ]:
# Вспомогательные функции (готовы, менять не нужно)
def bernoulli_pmf(n, m, p):
    return stats.binom.pmf(m, n, p)

def laplace_local(n, m, p):
    s = math.sqrt(n * p * (1 - p))
    return stats.norm.pdf((m - n * p) / s) / s

def poisson_pmf(n, m, p):
    return stats.poisson.pmf(m, n * p)

def plant_bags(n_bags, n_seeds, p):
    # Сажаем n_bags пакетов по n_seeds семян; каждое семя всходит с вероятностью p независимо.
    # Возвращает число взошедших семян в каждом пакете.
    sprouted = rng.random((n_bags, n_seeds)) < p
    return sprouted.sum(axis=1)

---
## Этап 1. Прогноз до запуска (1 балл)

Вы сажаете пакет из $n$ = `N_BAG` семян со всхожестью $p$ = `P_GERM`. Агроном спрашивает: «Сколько взойдёт? И сколько взойдёт в 95 % пакетов?»

**До запуска** кода посчитайте руками (или на калькуляторе):

1. наивероятнейшее число всходов $m_0$ по формуле $np - q \le m_0 \le np + p$;
2. интервал $np \pm 1{,}96\sqrt{npq}$, в который по интегральной формуле Лапласа попадут примерно 95 % пакетов.

**Прогноз (до запуска):**

- $m_0 = $ ...
- интервал 95 %: от ... до ...

In [ ]:
bags = plant_bags(10_000, N_BAG, P_GERM)

# TODO: найдите самое частое число всходов в симуляции (подсказка: np.bincount и argmax)
m0_sim = ...

# TODO: найдите долю пакетов, попавших в ВАШ интервал из прогноза (впишите свои границы)
low, high = ..., ...
share_in = ...

print("Самое частое число всходов в симуляции:", m0_sim)
print(f"Доля пакетов в интервале [{low}; {high}]: {share_in:.3f}")

m = np.arange(N_BAG + 1)
plt.bar(m, np.bincount(bags, minlength=N_BAG + 1) / len(bags), alpha=0.6, label="симуляция 10 000 пакетов")
plt.plot(m, bernoulli_pmf(N_BAG, m, P_GERM), "o-", color="black", label="Бернулли")
plt.axvspan(low - 0.5, high + 0.5, color="orange", alpha=0.15, label="ваш интервал")
plt.xlabel("число взошедших семян"); plt.legend(); plt.show()

**Что получилось:** ...

**Интерпретация.** Совпал ли прогноз с симуляцией? Если доля пакетов в интервале заметно отличается от 0,95 — почему? (Подумайте: сколько у вас семян, насколько $p$ близко к 1, симметрично ли распределение, может ли граница интервала выйти за $n$?)

...

---
## Этап 2. Где врёт Лаплас (1 балл)

Сравним точную формулу Бернулли и локальную формулу Лапласа для вашего $p$ при $n = 10, 30, 100$.

**Прогноз (до запуска):** при каком $n$ ошибка Лапласа будет больше? В какой части графика (слева от пика, на пике, справа) она будет максимальной и почему?

...

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, n in zip(axes, [10, 30, 100]):
    m = np.arange(n + 1)
    exact = bernoulli_pmf(n, m, P_GERM)
    approx = laplace_local(n, m, P_GERM)

    # TODO: максимальная абсолютная ошибка Лапласа и значение m, где она достигается
    max_err = ...
    worst_m = ...

    ax.bar(m, exact, alpha=0.5, label="Бернулли")
    ax.plot(m, approx, "r.-", label="Лаплас")
    ax.axvline(worst_m, color="gray", ls="--")
    ax.set_title(f"n = {n}: макс. ошибка {max_err:.4f} при m = {worst_m}")
    ax.legend()
plt.tight_layout(); plt.show()

**Что получилось:** ...

**Интерпретация.** Почему ошибка максимальна именно там? Как форма распределения Бернулли при вашем $p$ отличается от колокола Лапласа? Посчитайте $npq$ для каждого $n$ и свяжите с условием применимости формулы Лапласа из методички.

...

---
## Этап 3. Граница Пуассона (1 балл)

Вероятность того, что растение заражено болезнью, равна $p$. Насколько маленьким должно быть $p$, чтобы формулой Пуассона можно было пользоваться?

Ошибку будем мерить так: $d = \frac12\sum_m |P_\text{Бернулли}(m) - P_\text{Пуассон}(m)|$. Это *наибольшая ошибка в вероятности любого события*, которую можно получить, заменив Бернулли на Пуассона. Ваш порог — `EPS`.

**Прогноз (до запуска):** при каком $p$ ошибка станет больше `EPS`? Будет ли эта граница сильно зависеть от числа растений $n$ (50, 500 или 5000)?

...

In [ ]:
def poisson_distance(n, p):
    m = np.arange(n + 1)
    diff = np.abs(bernoulli_pmf(n, m, p) - poisson_pmf(n, m, p)).sum()
    return 0.5 * (diff + stats.poisson.sf(n, n * p))   # + «хвост» Пуассона за пределами n

ps = np.round(np.arange(0.001, 0.201, 0.001), 3)
plt.figure()
for n in [50, 500, 5000]:
    d = np.array([poisson_distance(n, p) for p in ps])

    # TODO: наибольшее p из ps, при котором ошибка d ещё меньше EPS
    p_border = ...

    plt.plot(ps, d, label=f"n = {n}: граница p ≈ {p_border}")
plt.axhline(EPS, color="red", ls="--", label=f"порог {EPS}")
plt.xlabel("p"); plt.ylabel("ошибка Пуассона d"); plt.legend(); plt.show()

**Что получилось:** ...

**Интерпретация.** От чего на самом деле зависит точность формулы Пуассона: от $n$ или от $p$? Сравните с формулировкой условий в методичке («$n$ велико, $p$ мало, $np < 10$»). Какое из условий главное? Можно ли угадать связь между найденной границей и вашим порогом `EPS`?

...

---
## Этап 4. Агроном-скептик (1 балл)

Ниже — данные полевого опыта вашего варианта: число взошедших семян в 60 пакетах по 50 семян. Заявленная всхожесть партии — `P_GERM`.

Вопрос: **соответствуют ли данные схеме Бернулли** с этой всхожестью? Если нет — какое из условий схемы нарушено: независимость испытаний, одинаковая вероятность $p$ в каждом испытании, или данные вообще не похожи на результат случайного опыта?

Подсказка: сравнивайте не только среднее, но и **разброс**. Для схемы Бернулли дисперсия числа успехов равна $npq$.

**Прогноз (до анализа):** что вы ожидаете увидеть, если данные честные? Каким будет среднее, какой разброс?

...

In [ ]:
# Данные полевого опыта: число взошедших семян в 60 пакетах по 50 семян (вариант → данные)
FIELD_DATA = {
    1: "35 36 36 35 35 35 36 35 36 36 36 37 37 35 35 36 35 36 37 37 37 36 36 37 35 37 36 36 36 35 35 37 36 37 37 35 35 37 35 36 36 36 37 36 36 35 36 36 37 36 36 37 37 37 37 35 36 37 35 36",
    2: "42 45 46 43 44 42 38 38 43 44 44 41 43 42 43 42 42 46 48 41 42 40 44 44 45 42 46 39 39 43 40 43 43 43 42 36 44 44 40 40 39 39 41 40 42 41 42 45 39 36 43 42 39 40 48 41 39 46 45 43",
    3: "32 33 33 31 31 31 31 31 33 33 33 32 33 31 33 31 31 31 32 33 31 32 33 32 32 32 32 32 33 32 31 32 33 31 31 33 31 31 32 33 31 33 33 33 31 32 33 33 33 32 32 31 31 31 32 32 33 33 31 31",
    4: "47 36 43 42 39 39 33 47 45 30 41 35 29 43 36 44 32 46 41 30 44 23 35 45 37 30 46 40 32 42 33 40 34 32 41 45 37 38 47 35 39 41 38 39 38 37 34 42 47 34 36 33 38 45 40 35 38 30 36 41",
    5: "44 45 45 45 44 43 45 45 43 43 44 44 44 43 44 43 43 43 44 44 44 43 45 43 45 44 43 45 43 43 43 43 44 44 44 45 45 44 45 44 44 43 43 45 45 43 43 44 44 45 44 44 43 44 44 45 45 45 44 44",
    6: "27 35 36 42 33 24 35 36 41 26 38 30 40 28 34 39 35 24 33 39 30 29 32 37 34 39 29 32 39 31 29 42 24 29 35 32 32 37 26 31 35 33 28 30 39 25 36 30 43 30 37 39 38 33 29 35 36 22 31 33",
    7: "39 39 40 39 40 40 41 39 41 39 41 40 39 39 40 40 41 41 39 39 41 40 40 39 41 40 40 40 41 41 40 39 40 41 40 41 40 39 41 39 41 39 40 41 41 40 39 41 40 41 39 41 40 39 40 40 41 39 39 41",
    8: "44 38 49 44 43 45 44 46 48 43 45 45 47 48 46 47 46 42 46 40 48 48 45 42 46 47 43 48 43 46 48 44 44 48 42 43 48 46 47 45 40 45 43 43 49 47 46 47 44 44 44 45 49 44 48 45 47 47 43 42",
    9: "35 37 37 37 37 37 36 35 35 35 35 36 36 37 37 36 37 36 35 35 37 36 35 36 37 37 35 35 37 37 35 37 36 37 35 36 37 37 35 37 36 35 37 37 37 37 36 36 35 37 37 36 35 37 35 37 36 37 35 35",
    10: "41 44 41 40 41 40 41 38 40 42 43 42 41 42 38 40 39 44 40 44 44 45 41 41 40 41 41 40 40 37 37 38 42 44 41 41 44 36 40 43 41 43 44 41 43 37 42 39 47 46 43 39 41 43 43 44 44 43 40 45",
    11: "21 27 27 38 27 35 35 30 30 27 21 29 25 26 36 33 35 35 23 30 38 35 37 28 39 33 35 28 38 36 39 30 34 27 25 30 31 40 26 40 36 35 27 30 31 26 34 39 35 25 33 36 28 38 38 21 37 34 39 27",
    12: "40 36 34 35 41 39 32 38 34 24 29 28 26 32 33 38 37 41 34 35 26 28 36 41 35 33 37 29 35 40 42 45 46 39 35 30 39 30 31 42 37 32 38 43 40 36 38 37 47 28 29 31 29 32 44 41 37 32 39 33",
    13: "42 42 43 42 44 46 47 43 43 45 47 47 41 47 41 42 48 46 45 42 43 42 41 44 42 44 45 45 42 44 39 43 43 38 44 41 42 40 44 38 45 42 45 45 45 42 49 46 42 37 41 44 39 41 43 43 41 41 40 41",
    14: "36 31 22 34 31 36 32 30 29 34 36 32 28 38 35 32 33 32 29 33 45 37 45 33 41 34 31 37 36 39 37 34 41 40 31 27 33 36 33 38 26 36 27 32 40 24 33 33 42 23 35 32 23 42 31 34 40 32 32 35",
    15: "39 40 38 35 40 38 39 36 41 37 38 39 42 36 33 39 35 39 41 40 41 35 35 36 38 40 42 38 41 38 37 38 40 35 37 38 40 40 37 37 41 40 38 41 36 42 36 39 33 35 42 38 41 43 40 39 38 41 43 41",
    16: "43 44 44 44 44 44 43 44 45 45 44 44 43 43 44 43 43 43 44 43 44 43 43 43 44 43 45 43 44 44 43 44 44 45 43 44 45 43 44 43 43 43 44 43 45 43 45 44 43 45 44 45 44 45 43 45 43 45 45 44",
    17: "38 30 41 31 39 29 31 38 42 32 45 30 33 34 31 41 32 27 39 40 37 26 32 17 37 28 26 24 39 37 41 41 42 41 36 34 37 37 29 34 28 39 23 39 33 33 39 42 33 37 34 35 32 30 31 32 33 36 31 45",
    18: "41 41 39 39 40 39 39 40 40 41 39 41 41 41 41 41 40 41 41 41 40 39 40 41 39 39 39 39 40 40 40 39 41 40 40 39 40 41 40 40 40 41 39 40 39 39 39 41 39 39 41 40 40 40 39 41 41 41 39 40",
    19: "47 50 45 44 45 44 45 46 46 46 41 45 45 47 45 46 46 46 45 48 48 43 48 47 45 40 46 41 45 48 44 48 45 47 46 48 46 42 47 44 48 43 41 47 42 45 45 45 45 44 45 44 44 48 47 47 49 46 47 45",
    20: "37 36 35 35 37 37 37 36 35 35 37 35 35 37 37 37 35 35 36 35 35 35 37 36 37 37 37 35 36 36 36 36 35 36 36 35 35 36 35 36 35 37 36 35 36 35 37 37 36 37 37 37 36 36 37 37 36 36 36 36",
    21: "42 43 43 41 42 43 41 43 43 43 42 42 42 41 42 43 43 43 42 42 43 42 42 41 42 43 41 41 43 41 41 41 43 41 43 42 43 42 42 43 41 43 43 41 42 42 41 43 43 42 43 41 41 41 43 42 43 43 41 42",
    22: "31 33 35 35 27 34 36 34 32 28 36 35 28 33 36 31 35 28 26 32 33 35 33 25 33 38 33 35 30 28 34 26 30 36 37 34 32 32 30 31 37 35 25 28 28 28 31 35 29 29 38 39 29 31 32 28 35 30 32 36",
    23: "44 28 30 27 39 43 36 43 34 36 41 42 43 38 36 49 40 44 32 41 29 42 32 41 27 37 27 39 37 40 36 44 40 34 32 36 39 42 37 42 35 40 42 27 36 28 42 41 29 36 37 41 43 41 43 39 35 43 35 27",
    24: "40 39 46 44 34 45 46 48 45 41 39 49 43 47 46 42 40 38 37 42 46 32 47 47 45 40 48 47 39 42 41 38 39 43 43 43 47 46 43 48 39 50 40 46 49 47 43 46 46 41 47 36 47 36 49 43 36 42 40 49",
    25: "30 30 33 25 35 30 37 36 35 32 36 39 35 30 35 29 30 31 36 31 34 29 34 30 33 33 28 32 27 36 34 36 33 38 29 33 36 36 37 33 37 37 36 30 29 30 31 31 40 34 33 31 30 36 34 35 34 34 34 31",
    26: "39 44 32 44 47 35 41 39 44 43 44 42 35 41 42 43 35 40 43 45 42 45 39 30 39 31 37 35 42 40 39 39 48 45 37 39 44 43 44 42 39 31 38 36 35 32 40 43 40 37 41 45 43 36 34 36 32 32 36 34",
    27: "45 47 46 45 41 43 48 45 47 41 43 47 44 46 46 44 44 47 46 43 43 47 44 40 43 45 43 41 45 45 45 48 45 42 42 44 47 46 45 46 44 46 44 46 47 47 42 42 49 45 45 46 45 46 45 41 44 44 43 43",
    28: "37 29 36 34 36 34 36 34 37 30 39 36 39 40 34 35 34 41 28 36 32 33 35 38 36 35 41 34 36 29 37 33 35 36 36 37 33 39 36 37 37 39 31 33 35 34 38 35 33 33 37 38 38 35 38 35 32 34 31 34",
    29: "40 41 40 41 39 40 39 41 39 40 41 41 40 41 39 41 41 41 39 41 39 40 40 39 40 41 39 39 39 40 41 39 41 40 40 40 41 41 40 39 39 40 39 40 40 39 40 39 39 39 40 40 40 41 40 40 41 41 40 39",
    30: "31 25 34 36 36 39 24 25 28 41 38 27 24 28 31 35 30 38 32 25 33 32 30 20 30 31 23 32 33 32 21 31 35 38 25 24 34 28 21 34 30 34 37 32 25 30 38 40 35 33 27 27 28 34 34 38 35 30 32 35",
}

In [ ]:
field = np.array(FIELD_DATA[VARIANT].split(), dtype=int)
n_seeds, n_bags = 50, len(field)

print("Ваши данные:", field)
print(f"Среднее: {field.mean():.2f}   (по теории np = {n_seeds * P_GERM:.2f})")
print(f"Выборочная дисперсия: {field.var(ddof=1):.2f}   (по теории npq = {n_seeds * P_GERM * (1 - P_GERM):.2f})")

In [ ]:
# Симулируем 2000 «честных» полевых опытов такого же размера
honest_vars = []
for _ in range(2000):
    fake_field = plant_bags(n_bags, n_seeds, P_GERM)
    # TODO: добавьте в список выборочную дисперсию этого честного опыта
    ...

honest_vars = np.array(honest_vars)
# TODO: доля честных опытов, у которых дисперсия получилась не меньше / не больше вашей
share_ge = ...
share_le = ...
print(f"Честных опытов с дисперсией ≥ вашей: {share_ge:.3f};  ≤ вашей: {share_le:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(honest_vars, bins=40, alpha=0.6, label="2000 честных опытов")
axes[0].axvline(field.var(ddof=1), color="red", lw=2, label="ваши данные")
axes[0].set_xlabel("выборочная дисперсия"); axes[0].legend()
m = np.arange(n_seeds + 1)
axes[1].bar(m, np.bincount(field, minlength=n_seeds + 1) / n_bags, alpha=0.6, label="ваши данные")
axes[1].plot(m, bernoulli_pmf(n_seeds, m, P_GERM), "o-", color="black", label="Бернулли")
axes[1].set_xlim(n_seeds * P_GERM - 20, n_seeds * P_GERM + 20); axes[1].legend()
plt.tight_layout(); plt.show()

**Что получилось:** ...

**Вердикт:** данные соответствуют / не соответствуют схеме Бернулли.

**Интерпретация.** На чём основан вердикт? Если схема нарушена — какое именно условие и какая реальная причина в поле или в лаборатории могла это вызвать? Почему вывод нельзя сделать по одному только среднему?

...

---
## Этап 5. Наши монетки (1 балл)

Внесите свои 5 серий по 10 бросков с семинара и таблицу всей группы с доски.

**Прогноз (до запуска):** можно ли по вашим 50 броскам заподозрить, что монета нечестная? А по броскам всей группы?

...

In [ ]:
# TODO: число орлов в каждой из ваших 5 серий (с листочка)
my_series = [..., ..., ..., ..., ...]

# TODO: палочки с доски (сколько серий группы дали m орлов)
group_tally = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0}

my_heads, my_tosses = sum(my_series), 10 * len(my_series)
g = np.array([group_tally[k] for k in range(11)])
group_heads, group_tosses = int((g * np.arange(11)).sum()), int(10 * g.sum())

def how_unusual(heads, tosses, n_sim=10_000):
    # Доля виртуальных экспериментов с честной монетой, где отклонение от половины не меньше нашего
    sim = rng.binomial(tosses, 0.5, size=n_sim)
    return np.mean(np.abs(sim - tosses / 2) >= abs(heads - tosses / 2))

print(f"Вы: {my_heads} орлов из {my_tosses}, доля {my_heads / my_tosses:.3f}; "
      f"так же или более необычно у честной монеты: {how_unusual(my_heads, my_tosses):.1%}")
if group_tosses == 0:
    print("Группа: впишите палочки с доски в group_tally.")
else:
    print(f"Группа: {group_heads} орлов из {group_tosses}, доля {group_heads / group_tosses:.3f}; "
          f"так же или более необычно: {how_unusual(group_heads, group_tosses):.1%}")

**Что получилось:** ...

**Интерпретация.** Насколько необычны ваши результаты и результаты группы? Что бы вы ответили человеку, который по своим 5 сериям утверждает, что его монета «любит орла»? Оцените по интегральной формуле Лапласа: какое отклонение доли орлов от 0,5 можно было бы уверенно заметить на ваших 50 бросках и на бросках всей группы?

...